# FarmerVision — Serve & Test on Colab (free T4)

Runs the whole gateway on a free Colab GPU and exposes a **public API** via cloudflared:
**Qdrant vector DB + BGE-M3 retrieval + distilled Gemma generation + ViT vision**, with
guardrail + `live_data` injection (mandi / weather / yield come from your app).

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`. Your model artifacts must
already be in your GCS bucket (from `upload_artifacts_from_colab.py`).

Run the cells **top to bottom**. Cell 5 prints your public URL; cells 7-8 test everything.
The API stays up while this tab/runtime is alive (that's the "temporary" part).

## 1. Install dependencies

In [ ]:
!pip -q install qdrant-client sentence-transformers "transformers>=4.44" peft accelerate bitsandbytes fastapi uvicorn nest_asyncio kagglehub python-multipart >/dev/null 2>&1
print("deps installed")

## 2. Settings + sign in + pull artifacts from the bucket
Edit `HF_TOKEN` (gated Gemma) and `API_KEY`. `PROJECT` / `BUCKET` are prefilled.

In [ ]:
import torch, os
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU, then rerun this cell"
from google.colab import auth; auth.authenticate_user()

PROJECT = "project-7f232935-b8f7-4aad-881"        # your GCP project id
BUCKET  = "gs://farmervision-prod-artifacts"      # your GCS bucket
os.environ["HF_TOKEN"] = "hf_your_token_here"     # <-- PASTE your Hugging Face token
API_KEY = "change-me-to-a-secret"                 # <-- your app sends this in the X-API-Key header

!gcloud config set project {PROJECT} -q
!mkdir -p /content/art
!gcloud storage cp -r {BUCKET}/* /content/art/    # ~12 GB, a few minutes
print("pulled:")
!ls /content/art

## 3. Start Qdrant + restore the snapshot
Downloads the static **musl** build (no GLIBC issues) and restores your 723k-vector snapshot.

In [ ]:
import os, subprocess, tarfile, time, requests, json, glob

QDRANT_STORAGE = "/content/qdrant_storage"
QDRANT_URL     = "http://localhost:6333"
QDRANT_BIN     = "/content/qdrant"
os.makedirs(QDRANT_STORAGE, exist_ok=True)

def qdrant_alive(url=QDRANT_URL, timeout=1):
    try: return requests.get(f"{url}/readyz", timeout=timeout).ok
    except Exception: return False

def binary_ok(path=QDRANT_BIN):
    if not os.path.exists(path): return False
    try: return subprocess.run([path, "--version"], capture_output=True, timeout=60).returncode == 0
    except Exception: return False

if not qdrant_alive():
    if not binary_ok():
        if os.path.exists(QDRANT_BIN): os.remove(QDRANT_BIN)
        rel = requests.get("https://api.github.com/repos/qdrant/qdrant/releases/latest", timeout=30).json()
        asset = None
        for suffix in ("x86_64-unknown-linux-musl.tar.gz", "x86_64-unknown-linux-gnu.tar.gz"):
            asset = next((a for a in rel["assets"] if a["name"].endswith(suffix)), None)
            if asset: break
        if asset is None: raise RuntimeError("no linux x86_64 asset in " + rel["tag_name"])
        print(f"downloading qdrant {rel['tag_name']} ({asset['name']}) ...")
        open("/content/q.tar.gz", "wb").write(requests.get(asset["browser_download_url"], timeout=600).content)
        with tarfile.open("/content/q.tar.gz") as t:
            try: t.extractall("/content", filter="data")
            except TypeError: t.extractall("/content")
        os.chmod(QDRANT_BIN, 0o755)
        if not binary_ok(): raise RuntimeError("downloaded qdrant will not execute here")
    env = dict(os.environ, QDRANT__STORAGE__STORAGE_PATH=QDRANT_STORAGE, QDRANT__TELEMETRY_DISABLED="true")
    proc = subprocess.Popen([QDRANT_BIN], env=env, cwd="/content",
                            stdout=open("/content/qdrant.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(90):
        if qdrant_alive(): break
        if proc.poll() is not None:
            raise RuntimeError("qdrant exited early:\n" + open("/content/qdrant.log").read()[-1500:])
        time.sleep(1)
    else:
        raise RuntimeError("qdrant not ready in 90s - see /content/qdrant.log")
print("Qdrant ready on :6333")

man = json.load(open(glob.glob("/content/art/**/manifest.json", recursive=True)[0]))
COLLECTION = man["collection"]
existing = requests.get(f"{QDRANT_URL}/collections", timeout=30).json()["result"]["collections"]
if any(c["name"] == COLLECTION for c in existing):
    print(f"collection '{COLLECTION}' already present - skipping restore")
else:
    snap = glob.glob(f"/content/art/**/{man['snapshot']}", recursive=True)[0]
    print(f"uploading snapshot ({os.path.getsize(snap)/1e9:.2f} GB) - a few minutes ...")
    with open(snap, "rb") as fh:
        requests.post(f"{QDRANT_URL}/collections/{COLLECTION}/snapshots/upload?priority=snapshot",
                      files={"snapshot": (man["snapshot"], fh)}, timeout=7200).raise_for_status()

from qdrant_client import QdrantClient
qc = QdrantClient(url=QDRANT_URL, timeout=600)
print("points:", qc.get_collection(COLLECTION).points_count)

## 4. Load the embedder + generator
Prefers the **merged** Gemma (adapter baked in = fast); falls back to base + LoRA adapter.

In [ ]:
import glob, os, torch
from sentence_transformers import SentenceTransformer
from transformers import (AutoConfig, AutoTokenizer, AutoModelForCausalLM,
                          AutoModelForImageTextToText, BitsAndBytesConfig)

EMB = SentenceTransformer(man["embed_model"], device="cuda")

merged  = next((c for c in glob.glob("/content/art/**/merged", recursive=True)
                if os.path.isfile(os.path.join(c, "config.json"))), None)
adapter = None if merged else next(iter(glob.glob("/content/art/**/best_adapter", recursive=True)), None)
src     = merged or "google/gemma-3-4b-it"
tok_src = merged or adapter or "google/gemma-3-4b-it"
tokn    = os.environ["HF_TOKEN"]

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
cfg = AutoConfig.from_pretrained(src, token=tokn)
mm  = hasattr(cfg, "vision_config") or "text_config" in (getattr(cfg, "sub_configs", None) or {})
Cls = AutoModelForImageTextToText if mm else AutoModelForCausalLM
mdl = Cls.from_pretrained(src, quantization_config=bnb, device_map="cuda", token=tokn)
if adapter:
    from peft import PeftModel; mdl = PeftModel.from_pretrained(mdl, adapter)
tok = AutoTokenizer.from_pretrained(tok_src, token=tokn)
mdl.eval(); print("generator ready:", src)

## 5. Pipeline + API + public URL
Retrieval (tiers + fusion from the manifest), grounded generation with `live_data` injection,
FastAPI, and a cloudflared tunnel. **Prints your public URL at the end.**

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from typing import Optional
import uvicorn, threading, nest_asyncio, subprocess, re, torch, time

FW, TF, TG = man["fusion_weights"], man["tiers"]["fallback"], man["tiers"]["grounded"]
TOPK = man.get("top_k_default", 5)
SYS = ("You are FarmerVision, an assistant for Indian farmers. Answer ONLY from the numbered "
       "context [1],[2] and any LIVE DATA block; cite context facts like [1]. LIVE DATA "
       "(mandi prices/weather/yield) is authoritative - use those exact values. Reply in the "
       "question's language/script. Never invent numbers. If nothing covers it, say you don't "
       "have enough info and suggest the local KVK.")

def retrieve(query, intent="general"):
    w = FW.get(intent, FW["general"]); qv = EMB.encode(query, normalize_embeddings=True).tolist(); hits = []
    for st in ["pdf", "kcc"]:
        for h in qc.query_points(collection_name=COLLECTION, query=qv, limit=TOPK, with_payload=True,
                query_filter=Filter(must=[FieldCondition(key="source_type", match=MatchValue(value=st))])).points:
            hits.append({"raw": float(h.score), "fused": float(h.score)*w.get(st, 1.0),
                         "text": h.payload.get("text", ""), "src": st})
    hits.sort(key=lambda x: x["fused"], reverse=True); hits = hits[:TOPK]
    best = max((h["raw"] for h in hits), default=0.0)
    return ("grounded" if best >= TG else "fallback" if best >= TF else "abstain"), best, hits

@torch.inference_mode()
def generate(query, hits, live=None):
    ctx = "\n\n".join(f"[{i}] {h['text']}" for i, h in enumerate(hits, 1)) or "(no context)"
    parts = []
    if live: parts.append("LIVE DATA (authoritative):\n" + "\n".join(f"- {k}: {v}" for k, v in live.items()))
    parts.append(f"Context:\n{ctx}\n\nQuestion: {query}")
    msgs = [{"role": "user", "content": SYS + "\n\n" + "\n\n".join(parts)}]
    # NOTE: Gemma-3's apply_chat_template returns a dict, not a tensor -> handle it.
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    input_ids = enc["input_ids"].to(mdl.device)
    attn = enc.get("attention_mask"); attn = attn.to(mdl.device) if attn is not None else None
    out = mdl.generate(input_ids=input_ids, attention_mask=attn, max_new_tokens=400,
                       do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

app = FastAPI()
class Q(BaseModel):
    query: str; intent: str = "general"; live_data: Optional[dict] = None; skip_retrieval: bool = False

@app.get("/health")
def health(): return {"ok": True, "points": qc.get_collection(COLLECTION).points_count}

@app.post("/query")
def query(b: Q, x_api_key: Optional[str] = Header(None)):
    if x_api_key != API_KEY: raise HTTPException(401, "bad X-API-Key")
    if b.skip_retrieval: tier, best, hits = "skipped", 0.0, []
    else: tier, best, hits = retrieve(b.query, b.intent)
    if tier == "abstain" and not b.live_data:
        return {"tier": tier, "answer": None, "message": "no relevant info - try rephrasing"}
    return {"tier": tier, "top_score": round(best, 4), "answer": generate(b.query, hits, b.live_data),
            "sources": [{"n": i+1, "score": round(h["raw"], 4), "src": h["src"]} for i, h in enumerate(hits)]}

nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
                 daemon=True).start()
time.sleep(3)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cf
!chmod +x /content/cf
p = subprocess.Popen(["/content/cf", "tunnel", "--url", "http://localhost:8000"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    m = re.search(r"https://[-\w.]+\.trycloudflare\.com", line)
    if m: print("\nPUBLIC API URL:", m.group(0)); break

## 6. Add vision (`/vision` + `/diagnose`)
Loads the ViT the E2E eval uses (kagglehub + its `predict.py`) and adds the endpoints live.

In [ ]:
import kagglehub, sys, os, tempfile
from pathlib import Path
kagglehub.login()   # Kaggle username + token

vit_dir   = Path(kagglehub.dataset_download("iitm21f1003346/vits16-crop-disease"))
model_dir = next(vit_dir.rglob("predict.py")).parent
sys.path.insert(0, str(model_dir))
from predict import CropDiseaseModel
vit = CropDiseaseModel(device="cpu")          # CPU keeps T4 VRAM free for Gemma
print("ViT ready. classes:", vit.classes[:4], "...")

_samp = next((p for p in vit_dir.rglob("*.jpg")), None) or next((p for p in vit_dir.rglob("*.png")), None)
if _samp: print("self-test:", vit.predict(str(_samp)))

from fastapi import UploadFile, File
def _save(data):
    f = tempfile.NamedTemporaryFile(suffix=".jpg", delete=False); f.write(data); f.close(); return f.name

@app.post("/vision")
async def vision_ep(file: UploadFile = File(...), x_api_key: str = Header(None)):
    if x_api_key != API_KEY: raise HTTPException(401, "bad X-API-Key")
    p = _save(await file.read())
    try: r = vit.predict(p)
    finally: os.remove(p)
    return {"label": r["label"], "confidence": round(float(r["confidence"]), 4),
            "rejected": bool(r.get("rejected", False)), "top3": r.get("top3")}

@app.post("/diagnose")
async def diagnose_ep(file: UploadFile = File(...), x_api_key: str = Header(None)):
    if x_api_key != API_KEY: raise HTTPException(401, "bad X-API-Key")
    p = _save(await file.read())
    try: r = vit.predict(p)
    finally: os.remove(p)
    if r.get("rejected") or not r.get("label"):
        return {"diagnosis": r, "answer": None, "message": "image rejected / not a recognised leaf"}
    crop, disease = r["label"].split("__")[0], r["label"].replace("__", " ").replace("_", " ")
    tier, best, hits = retrieve(f"{disease} treatment and management", intent="field_practice")
    q = f"My {crop} crop likely has {disease}. Give the treatment from the context."
    return {"diagnosis": {"crop": crop, "disease": disease, "confidence": round(float(r['confidence']), 4)},
            "tier": tier, "answer": generate(q, hits) if hits else None,
            "sources": [{"n": i+1, "score": round(h["raw"], 4), "src": h["src"]} for i, h in enumerate(hits)]}

print("added /vision and /diagnose")

## 7. Test — text (RAG, live_data injection, guardrail)

In [ ]:
import requests, textwrap
BASE, KEY = "http://localhost:8000", API_KEY
print("HEALTH:", requests.get(f"{BASE}/health").json(), "\n")

tests = [
    {"query": "gehu me yellow rust ke liye kaunsi dawa daalein", "intent": "field_practice"},
    {"query": "how to control brown planthopper in paddy", "intent": "field_practice"},
    {"query": "paddy me kitna urea daalna chahiye", "intent": "field_practice"},
    {"query": "aaj gehu bech du ya rukun?", "skip_retrieval": True,
     "live_data": {"mandi_prices": "Wheat @ Varanasi: Rs 2480/quintal (2026-08-14)",
                   "weather": "dry, no rain next 3 days"}},
    {"query": "who won the cricket match yesterday"},
]
for t in tests:
    d = requests.post(f"{BASE}/query", headers={"X-API-Key": KEY}, json=t).json()
    print("=" * 80); print("Q:", t["query"])
    print("tier:", d.get("tier"), "| top_score:", d.get("top_score"), "| sources:", len(d.get("sources") or []))
    print("A:", textwrap.fill(d.get("answer") or d.get("message") or "", 100)[:800])

## 8. Test — vision (`/vision` + `/diagnose`)
Finds a leaf image automatically; if none, prints a one-line upload fallback.

In [ ]:
import requests, json, glob
from pathlib import Path

img = None
_s = globals().get("_samp")
if _s: img = str(_s)
if not img:
    for base in [str(vit_dir), "/content/art", "/content"]:
        hits = [p for e in ("jpg", "jpeg", "png")
                for p in glob.glob(f"{base}/**/*.{e}", recursive=True) if "qdrant" not in p]
        if hits: img = hits[0]; break

if not img:
    print("No image found. In a NEW cell run:  from google.colab import files; files.upload()")
    print("pick any crop-leaf photo, then re-run THIS cell (it reads /content).")
else:
    print("using image:", img, "\n")
    print("vit.predict ->", vit.predict(img), "\n")
    for ep in ["/vision", "/diagnose"]:
        with open(img, "rb") as f:
            d = requests.post(f"http://localhost:8000{ep}", headers={"X-API-Key": API_KEY},
                              files={"file": (Path(img).name, f, "image/jpeg")}).json()
        print("=" * 80); print(ep); print(json.dumps(d, indent=2, ensure_ascii=False)[:1200])